# Engenharia de Atributos: Risco de Lesão no Futebol

Este notebook realiza engenharia de atributos e preparação de dados para o treinamento do modelo.

**Objetivos:**
- Carregar dataset pré processado e aplicar rótulos de risco
- Codificar features categóricas (posição via one-hot)
- Selecionar features relevantes para modelagem
- Criar features derivadas (interações e agregações)
- Salvar dataset processado para treinamento

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent / "src"))
from preprocessing import load_and_preprocess
from risk_score import engineer_risk_label

print("Imports concluídos")

Imports concluídos


## 1. Carregar e Aplicar Rótulos de Risco

In [2]:
print("Carregando e pré processando dados...")
data_dir = Path.cwd().parent
df = load_and_preprocess(data_dir)
df = engineer_risk_label(df)

print(f"Dados carregados: {df.shape}")
print(f"Rótulos de risco: {df['risk_label'].value_counts().to_dict()}")

Carregando e pré processando dados...
Dados carregados: (28, 59)
Rótulos de risco: {'Medium': 16, 'High': 7, 'Low': 5}


## 2. Seleção de Features

In [3]:
# Define features principais para modelagem
health_features = [
    'general_health', 'energy_level', 'sleep_hours', 'sleep_quality', 
    'wakes_rested', 'stress_level', 'motivation'
]

post_features = [
    'rpe', 'fatigue_level', 'post_discomfort'
]

match_features = [
    'accurate_pass', 'missed_pass', 'interception', 'shot_on_goal',
    'successful_dribble', 'tackle_won', 'foul_made', 'goal'
]

# Verifica disponibilidade das features
available_health = [f for f in health_features if f in df.columns]
available_post = [f for f in post_features if f in df.columns]
available_match = [f for f in match_features if f in df.columns]

print(f"Features de saúde disponíveis: {len(available_health)}")
print(f"Features pós-atividade disponíveis: {len(available_post)}")
print(f"Features de partida disponíveis: {len(available_match)}")
print(f"Total de features selecionadas: {len(available_health) + len(available_post) + len(available_match)}")

Features de saúde disponíveis: 7
Features pós-atividade disponíveis: 3
Features de partida disponíveis: 8
Total de features selecionadas: 18


## 3. Criar Matriz de Features

In [4]:
# Seleciona as features
feature_cols = available_health + available_post + available_match
X = df[feature_cols].copy()
y = df['risk_label'].copy()

print(f"Dimensão da matriz de features: {X.shape}")
print(f"Dimensão do alvo: {y.shape}")
print(f"\nTipos das features:")
print(X.dtypes.value_counts())
print(f"\nDistribuição do alvo:")
print(y.value_counts())

Dimensão da matriz de features: (28, 18)
Dimensão do alvo: (28,)

Tipos das features:
int64      9
float64    9
Name: count, dtype: int64

Distribuição do alvo:
risk_label
Medium    16
High       7
Low        5
Name: count, dtype: int64


## 4. Tratar One-Hot Encoding para Posição

In [5]:
# Adiciona posição como features one-hot
position_dummies = pd.get_dummies(df['position'], prefix='pos', drop_first=True)
X = pd.concat([X.reset_index(drop=True), position_dummies.reset_index(drop=True)], axis=1)

print(f"Matriz de features após codificação de posição: {X.shape}")
print(f"Categorias de posição: {df['position'].unique()}")
print(f"\nNovas colunas de feature: {list(position_dummies.columns)}")

Matriz de features após codificação de posição: (28, 19)
Categorias de posição: <StringArray>
['Goleiro', 'Linha']
Length: 2, dtype: str

Novas colunas de feature: ['pos_Linha']


## 5. Verificar Valores Ausentes Restantes

In [6]:
print(f"Valores ausentes na matriz de features:")
missing = X.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("Sem valores ausentes")

print(f"\nEstatísticas da matriz de features:")
print(X.describe().round(2))

Valores ausentes na matriz de features:
Sem valores ausentes

Estatísticas da matriz de features:
       general_health  energy_level  sleep_hours  sleep_quality  wakes_rested  \
count           28.00         28.00        28.00          28.00         28.00   
mean             2.82          2.25         6.79           2.43          1.89   
std              0.77          0.75         0.69           0.50          0.50   
min              2.00          1.00         5.00           2.00          1.00   
25%              2.00          2.00         6.00           2.00          2.00   
50%              3.00          2.00         7.00           2.00          2.00   
75%              3.00          3.00         7.00           3.00          2.00   
max              4.00          3.00         8.00           3.00          3.00   

       stress_level  motivation    rpe  fatigue_level  post_discomfort  \
count         28.00        28.0  28.00          28.00            28.00   
mean           1.89     

## 6. Escalonamento de Features

In [7]:
# Padroniza features numéricas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Features padronizadas")
print(f"\nEstatísticas das features escalonadas (esperado media=0, desvio=1):")
print(X_scaled.describe().round(3))

Features padronizadas

Estatísticas das features escalonadas (esperado media=0, desvio=1):
       general_health  energy_level  sleep_hours  sleep_quality  wakes_rested  \
count          28.000        28.000       28.000         28.000        28.000   
mean           -0.000        -0.000        0.000          0.000         0.000   
std             1.018         1.018        1.018          1.018         1.018   
min            -1.083        -1.694       -2.650         -0.866        -1.828   
25%            -1.083        -0.339       -1.166         -0.866         0.219   
50%             0.235        -0.339        0.318         -0.866         0.219   
75%             0.235         1.016        0.318          1.155         0.219   
max             1.554         1.016        1.802          1.155         2.267   

       stress_level  motivation     rpe  fatigue_level  post_discomfort  \
count        28.000        28.0  28.000         28.000           28.000   
mean          0.000         0

## 7. Codificar Variável Alvo

In [8]:
# Codifica o alvo: Low=0, Medium=1, High=2
risk_encoding = {'Low': 0, 'Medium': 1, 'High': 2}
y_encoded = y.map(risk_encoding)

print(f"Variável alvo codificada")
print(f"\nCodificação: {risk_encoding}")
print(f"\nDistribuição do alvo codificado:")
print(y_encoded.value_counts().sort_index())

Variável alvo codificada

Codificação: {'Low': 0, 'Medium': 1, 'High': 2}

Distribuição do alvo codificado:
risk_label
0     5
1    16
2     7
Name: count, dtype: int64


## 8. Salvar Dataset Processado

In [9]:
# Cria diretorio data/processed se nao existir
processed_dir = data_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Salva matriz de features
X_scaled.to_csv(processed_dir / "X_scaled.csv", index=False)
y_encoded.to_csv(processed_dir / "y_encoded.csv", index=False, header=['risk_label'])

# Salva metadados
metadata = {
    'n_samples': len(X_scaled),
    'n_features': len(X_scaled.columns),
    'feature_names': X_scaled.columns.tolist(),
    'risk_encoding': risk_encoding,
    'scaler_params': {
        'mean': scaler.mean_.tolist(),
        'scale': scaler.scale_.tolist()
    }
}

import json
with open(processed_dir / "metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"X_scaled.csv: {X_scaled.shape}")
print(f"y_encoded.csv: {y_encoded.shape}")
print(f"metadata.json")

X_scaled.csv: (28, 19)
y_encoded.csv: (28,)
metadata.json


## 9. Resumo das Features

In [10]:
print("Resumo da Feature Engineering")
print(f"\nDimensão do dataset: {X_scaled.shape}")
print(f"Distribuição do alvo: {y_encoded.value_counts().sort_index().to_dict()}")
print(f"\nCategorias de features:")
print(f"Indicadores de saúde: {len(available_health)}")
print(f"Indicadores pós-atividade: {len(available_post)}")
print(f"Métricas de desempenho da partida: {len(available_match)}")
print(f"One-hot de posição: {len(position_dummies.columns)}")
print(f"Total: {X_scaled.shape[1]}")
print(f"\nPronto para a fase de treinamento do modelo")

Resumo da Feature Engineering

Dimensão do dataset: (28, 19)
Distribuição do alvo: {0: 5, 1: 16, 2: 7}

Categorias de features:
Indicadores de saúde: 7
Indicadores pós-atividade: 3
Métricas de desempenho da partida: 8
One-hot de posição: 1
Total: 19

Pronto para a fase de treinamento do modelo
